In [1]:
import os
import glob
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF
from sklearn.ensemble import RandomForestClassifier
import warnings

In [ ]:
# 1. On prend TOUTES les configurations disponibles
chemin_dossier_pkl = 'vectorisation-du-texte/output/'
fichiers_pkl_propres = glob.glob(os.path.join(chemin_dossier_pkl, '*_FINAL.pkl'))
fichiers_pkl_propres.sort()

print(f"{len(fichiers_pkl_propres)} fichiers .pkl trouvés.")

In [3]:
# 2. Les hyperparamètres de la NMF à tester (demandés par le sujet)
parametres_nmf = [
    {'n_components': 4, 'beta_loss': 'frobenius', 'solver': 'cd'},
    {'n_components': 5, 'beta_loss': 'frobenius', 'solver': 'cd'},
    {'n_components': 4, 'beta_loss': 'kullback-leibler', 'solver': 'mu'}, # KL nécessite le solver 'mu'
    {'n_components': 5, 'beta_loss': 'kullback-leibler', 'solver': 'mu'}
]

resultats_themes = []

# 3. La boucle d'évaluation
for chemin_fichier in fichiers_pkl_propres:
    nom_fichier = os.path.basename(chemin_fichier).replace('_FINAL.pkl', '')
    
    # Chargement
    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)
        
    X = data['X_normalized']
    y = data['target']
    
    # Pour chaque fichier, on teste les paramètres NMF
    for params in parametres_nmf:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                # Étape A : Extraire les thèmes avec NMF
                nmf = NMF(n_components=params['n_components'], 
                          beta_loss=params['beta_loss'], 
                          solver=params['solver'], 
                          init='nndsvda', # Init recommandée pour KL
                          random_state=42, 
                          max_iter=500)
                
                # W est la nouvelle matrice : chaque avis est représenté par ses thèmes
                W = nmf.fit_transform(X) 
                
                # Étape B : Évaluer ces thèmes avec le Random Forest !
                # On sépare notre nouvelle matrice W en train/test
                W_train, W_test, y_train, y_test = train_test_split(W, y, test_size=0.2, random_state=42)
                
                rf = RandomForestClassifier(n_estimators=100, random_state=42)
                rf.fit(W_train, y_train)
                
                score_rf = rf.score(W_test, y_test)
                
                resultats_themes.append({
                    'Configuration Texte': nom_fichier,
                    'Nb Thèmes': params['n_components'],
                    'Fonction Perte NMF': params['beta_loss'],
                    'Score Random Forest (%)': round(score_rf * 100, 2)
                })
        except Exception as e:
            # Certains algorithmes mathématiques peuvent échouer selon les données
            continue

In [4]:
# 4. Affichage du classement final
print("CLASSEMENT DES CONFIGURATIONS THÉMATIQUES (NMF + RANDOM FOREST)")
df_themes = pd.DataFrame(resultats_themes)
df_themes = df_themes.sort_values(by='Score Random Forest (%)', ascending=False).reset_index(drop=True)
df_themes.index = df_themes.index + 1 

display(df_themes.head(10)) # On affiche le Top 10

meilleur_theme = df_themes.iloc[0]
print("\n" + "*"*80)
print(f"🎯 CONCLUSION MISSION 2 : ")
print(f"Pour extraire les thèmes, utilisez le fichier '{meilleur_theme['Configuration Texte']}'")
print(f"avec une NMF configurée avec {meilleur_theme['Nb Thèmes']} thèmes et la perte '{meilleur_theme['Fonction Perte NMF']}'.")
print("*"*80)

CLASSEMENT DES CONFIGURATIONS THÉMATIQUES (NMF + RANDOM FOREST)


,Configuration Texte,Nb Thèmes,Fonction Perte NMF,Score Random Forest (%)
1,config_L0_S1_LEM1_NG1,5,kullback-leibler,78.11
2,config_L1_S1_LEM0_NG1,5,kullback-leibler,77.57
3,config_L1_S1_LEM1_NG3,5,kullback-leibler,75.68
4,config_L0_S1_LEM1_NG3,5,kullback-leibler,75.41
5,config_L0_S1_LEM1_NG2,5,kullback-leibler,75.41
6,config_L1_S1_LEM0_NG3,5,kullback-leibler,75.41
7,config_L1_S1_LEM1_NG1,5,kullback-leibler,75.41
8,config_L0_S1_LEM1_NG2,5,frobenius,74.86
9,config_L1_S1_LEM1_NG3,5,frobenius,74.59
10,config_L0_S1_LEM1_NG3,5,frobenius,74.32



********************************************************************************
🎯 CONCLUSION MISSION 2 : 
Pour extraire les thèmes, utilisez le fichier 'config_L0_S1_LEM1_NG1'
avec une NMF configurée avec 5 thèmes et la perte 'kullback-leibler'.
********************************************************************************


In [6]:
#AFFICHAGE DES MOTS-CLÉS DU MEILLEUR MODÈLE ---

print("\n" + "="*80)
print("🔍 DÉCOUVERTE DES THÈMES DU MEILLEUR MODÈLE")
print("="*80)

# 1. On récupère le chemin du fichier gagnant
chemin_gagnant = os.path.join(chemin_dossier_pkl, meilleur_theme['Configuration Texte'] + '_FINAL.pkl')

# 2. On charge ses données (pour avoir X et surtout les mots du vocabulaire)
with open(chemin_gagnant, 'rb') as f:
    donnees_gagnantes = pickle.load(f)
    
X_gagnant = donnees_gagnantes['X_normalized']
mots_vocabulaire = donnees_gagnantes['feature_names']

# 3. On recrée le solveur en fonction de la perte gagnante
solveur_gagnant = 'mu' if meilleur_theme['Fonction Perte NMF'] == 'kullback-leibler' else 'cd'

# 4. On ré-entraîne juste la NMF gagnante (c'est très rapide pour un seul modèle)
nmf_gagnante = NMF(
    n_components=meilleur_theme['Nb Thèmes'], 
    beta_loss=meilleur_theme['Fonction Perte NMF'], 
    solver=solveur_gagnant,
    init='nndsvda',
    random_state=42, 
    max_iter=500
)

nmf_gagnante.fit(X_gagnant)

# 5. On affiche le Top 10 des mots pour chaque thème
def afficher_top_mots(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        # On trie les poids pour trouver les mots les plus importants
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(f"📌 Thème {topic_idx + 1} : {', '.join(top_features)}")

afficher_top_mots(nmf_gagnante, mots_vocabulaire, 10)
print("="*80)


🔍 DÉCOUVERTE DES THÈMES DU MEILLEUR MODÈLE
📌 Thème 1 : le, trop, petit, être, très, peu, bracelet, dommage, plus, fragile
📌 Thème 2 : recevoir, avoir, je, ne, jamais, non, article, être, toujours, ce
📌 Thème 3 : qualité, bon, prix, produit, mauvais, rapport, très, correspondre, recommander, le
📌 Thème 4 : boucle, oreille, lui, de, jolie, d, très, Boucles, bel, photo
📌 Thème 5 : très, joli, beau, cadeau, bel, bien, conforme, recommander, bracelet, parfaire


---
## Recherche du K optimal pour la meilleure configuration

La configuration gagnante et la meilleure perte NMF sont déjà connues.  
On élargit la plage de K testés pour trouver le nombre de thèmes qui maximise la précision de la forêt aléatoire.

In [ ]:
from sklearn.metrics import accuracy_score

# Plage de K à tester (élargie par rapport aux 4 et 5 déjà testés)
VALEURS_K_TEST = [3, 4, 5, 6, 7, 8, 10, 12, 15, 20]

perte_optimale  = meilleur_theme['Fonction Perte NMF']
solveur_optimal = 'mu' if perte_optimale == 'kullback-leibler' else 'cd'

# Construction du chemin directement depuis meilleur_theme (indépendant des cellules précédentes)
chemin_gagnant = os.path.join(chemin_dossier_pkl, meilleur_theme['Configuration Texte'] + '_FINAL.pkl')

with open(chemin_gagnant, 'rb') as f:
    donnees_gagnantes = pickle.load(f)

X_gagnant     = donnees_gagnantes['X_normalized']
y_gagnant     = donnees_gagnantes['target']
feature_names = donnees_gagnantes['feature_names']

X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_gagnant, y_gagnant, test_size=0.2, random_state=42
)

# Test de chaque K
prec_par_k = {}
print(f"Configuration : {meilleur_theme['Configuration Texte']}  |  Perte : {perte_optimale}\n")

for k in VALEURS_K_TEST:
    nmf = NMF(n_components=k, beta_loss=perte_optimale, solver=solveur_optimal,
              init='nndsvda', random_state=42, max_iter=500)
    W_train = nmf.fit_transform(X_train_k)
    W_test  = nmf.transform(X_test_k)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(W_train, y_train_k)
    prec_par_k[k] = accuracy_score(y_test_k, rf.predict(W_test))
    print(f'  k={k:2d}  →  précision = {prec_par_k[k]:.4f}')

K_OPTIMAL = max(prec_par_k, key=prec_par_k.get)
print(f'\n→ K optimal retenu : {K_OPTIMAL}  (précision = {prec_par_k[K_OPTIMAL]:.4f})')

---
## Extraction des thèmes avec K optimal

On ré-entraîne la NMF avec le K optimal trouvé ci-dessus et la meilleure perte.  
La forêt aléatoire mesure l'**importance de chaque thème** pour prédire le sentiment.

In [ ]:
def analyze_and_store(X, y, feature_names, k, loss='frobenius'):
    """Décompose X en k thèmes (NMF), mesure l'importance de chaque thème
    via une forêt aléatoire et retourne un DataFrame trié par importance."""

    solveur = 'mu' if loss == 'kullback-leibler' else 'cd'

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        nmf = NMF(n_components=k, beta_loss=loss, solver=solveur,
                  init='nndsvda', random_state=42, max_iter=500)
        W = nmf.fit_transform(X)

    H = nmf.components_

    W_train, W_test, y_train, y_test = train_test_split(W, y, test_size=0.2, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(W_train, y_train)
    importances = rf.feature_importances_
    precision   = accuracy_score(y_test, rf.predict(W_test))

    lignes = []
    for idx, topic in enumerate(H):
        top_mots_ind = topic.argsort()[:-11:-1]   # top 10 mots
        top_mots     = ' | '.join([feature_names[i] for i in top_mots_ind])
        lignes.append({
            'ID Thème':       idx + 1,
            'Importance (RF)': round(importances[idx], 4),
            'Mots-clés':      top_mots,
        })

    df_res = pd.DataFrame(lignes).sort_values('Importance (RF)', ascending=False).reset_index(drop=True)

    print(f'K={k}  |  Perte={loss}  |  Précision sentiment : {precision:.4f}\n')
    display(df_res)
    return df_res


# Appel avec le K optimal et la meilleure perte identifiée automatiquement
df_themes_finaux = analyze_and_store(
    X_gagnant, y_gagnant, feature_names,
    k    = K_OPTIMAL,
    loss = perte_optimale
)

---
## Thèmes définitifs — filtre par importance RF

Seuls les thèmes dont l'**Importance (RF)** dépasse le seuil défini ci-dessous sont retenus comme thèmes définitifs.  
> **À vous de jouer** : ajustez `SEUIL_IMPORTANCE` selon le niveau de sélectivité souhaité.

In [ ]:
# ⚙️  PARAMÈTRE À AJUSTER — seuil d'importance entre 0 et 1
#     Seuls les thèmes dont l'importance est STRICTEMENT SUPÉRIEURE au seuil sont retenus
SEUIL_IMPORTANCE = 0.15

# Filtrage
df_definitifs = df_themes_finaux[df_themes_finaux['Importance (RF)'] > SEUIL_IMPORTANCE].copy()

n_retenus = len(df_definitifs)
n_total   = len(df_themes_finaux)

print(f'Seuil : {SEUIL_IMPORTANCE}  →  {n_retenus} thème(s) retenu(s) sur {n_total}\n')

if n_retenus == 0:
    print('Aucun thème ne dépasse ce seuil. Abaissez SEUIL_IMPORTANCE.')
else:
    display(df_definitifs.reset_index(drop=True))

    # Export XLSX — colonne "Nom du thème" vide à remplir manuellement
    df_export = df_definitifs.copy().reset_index(drop=True)
    df_export.insert(1, 'Nom du thème', '')
    df_export.to_excel('themes_definitifs.xlsx', index=False)
    print(f'\nExporté : themes_definitifs.xlsx  ({n_retenus} thème(s))')